[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/evals/evaluate_agent.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-assets/phoenix/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">评估 Agent</h1>

本 Notebook 以“与数据对话”的 Agent 为例，完整展示如何追踪和评估 Agent。

内容包括：
* 使用 Phoenix 装饰器手动为 Agent 插桩
* 使用 LLM 裁判评估函数调用准确性
* 通过对比真实标签评估函数调用准确性
* 评估 SQL 查询生成
* 评估 Python 代码生成
* 评估 Agent 的执行路径

## 安装依赖、导入库并设置 API 密钥

In [1]:
%pip install -q openai arize-phoenix openinference-instrumentation-openai python-dotenv duckdb openinference-instrumentation

Note: you may need to restart the kernel to use updated packages.


In [20]:
import dotenv

dotenv.load_dotenv()

import json
import os
from getpass import getpass

import duckdb
import pandas as pd
from IPython.display import Markdown
from openai import OpenAI
from openinference.instrumentation import (
    suppress_tracing,
)
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry.trace import StatusCode
from phoenix.otel import register
from pydantic import BaseModel, Field
from tqdm import tqdm

In [3]:
if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥：")
os.environ["OPENAI_API_KEY"] = openai_api_key

client = OpenAI()
model = "gpt-4o-mini"
project_name = "talk-to-your-data-agent"

# 启用 Phoenix 追踪

注册免费的 [Phoenix Cloud](https://app.phoenix.arize.com) 实例以获取 API 密钥。也可以选择[自行托管 Phoenix](https://arize.com/docs/phoenix/deployment)。

In [4]:
if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 请输入 Phoenix 收集器端点：")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint


if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥：")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key

In [5]:
tracer_provider = register(project_name=project_name, auto_instrument=True)

tracer = tracer_provider.get_tracer(__name__)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/phoenix/otel/otel.py:333: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: talk-to-your-data-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/schavali/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



## 准备数据集

Agent 将与本地数据库交互。首先加载数据：

In [21]:
store_sales_df = pd.read_parquet(
    "https://storage.googleapis.com/arize-phoenix-assets/datasets/unstructured/llm/llama-index/Store_Sales_Price_Elasticity_Promotions_Data.parquet"
)
store_sales_df.head()

,Store_Number,SKU_Coded,Product_Class_Code,Sold_Date,Qty_Sold,Total_Sale_Value,On_Promo
0,1320,6172800,22875,2021-11-02,3,56.849998,0
1,2310,6172800,22875,2021-11-03,1,18.950001,0
2,3080,6172800,22875,2021-11-03,1,18.950001,0
3,2310,6172800,22875,2021-11-06,1,18.950001,0
4,4840,6172800,22875,2021-11-07,1,18.950001,0


## 定义工具

现在可以定义 Agent 的工具。

### 工具 1：数据库查询

In [22]:
SQL_GENERATION_PROMPT = """
根据提示词生成 SQL 查询。除 SQL 查询外不要回复任何内容。
提示词：{prompt}

可用列：{columns}
表名：{table_name}
"""


def generate_sql_query(prompt: str, columns: list, table_name: str) -> str:
    """根据提示词生成 SQL 查询"""
    formatted_prompt = SQL_GENERATION_PROMPT.format(
        prompt=prompt, columns=columns, table_name=table_name
    )

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": formatted_prompt}],
    )

    return response.choices[0].message.content


@tracer.tool()
def lookup_sales_data(prompt: str) -> str:
    """使用 SQL 从 parquet 文件查询销售数据的实现"""
    try:
        table_name = "sales"
        # 将 parquet 文件读入 DuckDB 表
        duckdb.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM store_sales_df")

        print(store_sales_df.columns)
        print(table_name)
        sql_query = generate_sql_query(prompt, store_sales_df.columns, table_name)
        sql_query = sql_query.strip()
        sql_query = sql_query.replace("```sql", "").replace("```", "")

        with tracer.start_as_current_span(
            "execute_sql_query", openinference_span_kind="chain"
        ) as span:
            span.set_input(value=sql_query)

            # 执行 SQL 查询
            result = duckdb.sql(sql_query).df()
            span.set_output(value=str(result))
            span.set_status(StatusCode.OK)
        return result.to_string()
    except Exception as e:
        return f"访问数据出错： {str(e)}"

In [23]:
example_data = lookup_sales_data("Show me all the sales for store 1320 on November 1st, 2021")
example_data

Index(['Store_Number', 'SKU_Coded', 'Product_Class_Code', 'Sold_Date',
       'Qty_Sold', 'Total_Sale_Value', 'On_Promo'],
      dtype='object')
sales


'    Store_Number  SKU_Coded  Product_Class_Code  Sold_Date  Qty_Sold  Total_Sale_Value  On_Promo\n0           1320    6173050               22875 2021-11-01         1          4.990000         0\n1           1320    6174250               22875 2021-11-01         1          0.890000         0\n2           1320    6176200               22975 2021-11-01         2         99.980003         0\n3           1320    6176800               22800 2021-11-01         1         14.970000         0\n4           1320    6177250               22975 2021-11-01         1          6.890000         0\n5           1320    6177300               22800 2021-11-01         1          9.990000         0\n6           1320    6177350               22800 2021-11-01         2         16.980000         0\n7           1320    6177700               22875 2021-11-01         1          3.190000         0\n8           1320    6178000               22875 2021-11-01         2          6.380000         0\n9           1320   

### 工具 2：数据可视化

In [24]:
class VisualizationConfig(BaseModel):
    chart_type: str = Field(..., description="要生成的图表类型")
    x_axis: str = Field(..., description="X 轴列名")
    y_axis: str = Field(..., description="Y 轴列名")
    title: str = Field(..., description="Title of the chart")


@tracer.chain()
def extract_chart_config(data: str, visualization_goal: str) -> dict:
    """生成图表可视化配置。

    参数：
        data：包含待可视化数据的字符串。
        visualization_goal：描述可视化应展示的内容。

    返回：
        包含折线图配置的字典。
    """
    prompt = f"""根据以下数据生成图表配置：{data}
    展示目标：{visualization_goal}"""

    response = client.beta.chat.completions.parse(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format=VisualizationConfig,
    )

    try:
        # 从回复中提取坐标轴与标题信息
        content = response.choices[0].message.content

        # 返回结构化图表配置
        return {
            "chart_type": content.chart_type,
            "x_axis": content.x_axis,
            "y_axis": content.y_axis,
            "title": content.title,
            "data": data,
        }
    except Exception:
        return {
            "chart_type": "line",
            "x_axis": "date",
            "y_axis": "value",
            "title": visualization_goal,
            "data": data,
        }


@tracer.chain()
def create_chart(config: VisualizationConfig) -> str:
    """根据配置创建图表"""
    prompt = f"""根据以下配置编写创建图表的 Python 代码。
    仅返回代码，不要其他文本。
    config: {config}"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )

    code = response.choices[0].message.content
    code = code.replace("```python", "").replace("```", "")
    code = code.strip()

    return code


@tracer.tool()
def generate_visualization(data: str, visualization_goal: str) -> str:
    """根据数据与目标生成可视化"""
    config = extract_chart_config(data, visualization_goal)
    code = create_chart(config)
    return code

In [25]:
# code = generate_visualization(example_data, "A line chart of sales over each day in november.")

In [26]:
@tracer.tool()
def run_python_code(code: str) -> str:
    """在受限环境中执行 Python 代码"""
    # 创建包含绘图库的受限全局与局部字典
    restricted_globals = {
        "__builtins__": {
            "print": print,
            "len": len,
            "range": range,
            "sum": sum,
            "min": min,
            "max": max,
            "int": int,
            "float": float,
            "str": str,
            "list": list,
            "dict": dict,
            "tuple": tuple,
            "set": set,
            "round": round,
            "__import__": __import__,
            "json": __import__("json"),
        },
        "plt": __import__("matplotlib.pyplot"),
        "pd": __import__("pandas"),
        "np": __import__("numpy"),
        "sns": __import__("seaborn"),
    }

    try:
        # 在受限环境中执行代码
        exec_locals = {}
        exec(code, restricted_globals, exec_locals)

        # 捕获打印输出或返回图表
        exec_locals.get("__builtins__", {}).get("_", "")
        if "plt" in exec_locals:
            return exec_locals["plt"]

        # 返回前尝试将输出解析为 JSON
        return "代码执行成功"

    except Exception as e:
        return f"Error executing code: {str(e)}"

### 工具 3：数据分析

In [27]:
@tracer.tool()
def analyze_sales_data(prompt: str, data: str) -> str:
    """基于 AI 的销售数据分析实现"""
    # 根据分析类型和数据子集构建提示词
    prompt = f"""分析以下数据：{data}
    你的任务是回答以下问题：{prompt}"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )

    analysis = response.choices[0].message.content
    return analysis if analysis else "无法生成分析"

In [28]:
# analysis = analyze_sales_data("What is the most popular product SKU?", example_data)
# analysis

### 工具 Schema：

需要将工具描述传入 Agent 路由器。以下代码可便捷地完成此操作：

In [29]:
# 定义可供模型调用的工具或函数
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_sales_data",
            "description": "查询 Store Sales Price Elasticity Promotions 数据集中的数据",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {
                        "type": "string",
                        "description": "用户提供的原始提示词，不做修改。",
                    }
                },
                "required": ["prompt"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_sales_data",
            "description": "分析销售数据，提取洞察",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {
                        "type": "string",
                        "description": "lookup_sales_data 工具的输出。",
                    },
                    "prompt": {
                        "type": "string",
                        "description": "用户提供的原始提示词，不做修改。",
                    },
                },
                "required": ["data", "prompt"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "generate_visualization",
            "description": "生成用于数据可视化的 Python 代码",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {
                        "type": "string",
                        "description": "lookup_sales_data 工具的输出。",
                    },
                    "visualization_goal": {
                        "type": "string",
                        "description": "可视化目标。",
                    },
                },
                "required": ["data", "visualization_goal"],
            },
        },
    },
    # {
    #     "type": "function",
    #     "function": {
    #         "name": "run_python_code",
    #         "description": "在受限环境中运行 Python 代码",
    #         "parameters": {
    #             "type": "object",
    #             "properties": {
    #                 "code": {"type": "string", "description": "要运行的 Python 代码。"}
    #             },
    #             "required": ["code"]
    #         }
    #     }
    # }
]

# 将函数名映射到实现的字典
tool_implementations = {
    "lookup_sales_data": lookup_sales_data,
    "analyze_sales_data": analyze_sales_data,
    "generate_visualization": generate_visualization,
    # "run_python_code": run_python_code
}

## Agent 逻辑

定义工具后，即可定义 Agent 的主要路由和工具调用处理步骤。

In [30]:
@tracer.chain()
def handle_tool_calls(tool_calls, messages):
    for tool_call in tool_calls:
        function = tool_implementations[tool_call.function.name]
        function_args = json.loads(tool_call.function.arguments)
        result = function(**function_args)

        messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
    return messages

In [31]:
def start_main_span(messages):
    print("使用以下消息启动主 span：", messages)

    with tracer.start_as_current_span("AgentRun", openinference_span_kind="agent") as span:
        span.set_input(value=messages)
        ret = run_agent(messages)
        print("主 span 完成，返回值：", ret)
        span.set_output(value=ret)
        span.set_status(StatusCode.OK)
        return ret


def run_agent(messages):
    print("使用以下消息运行 Agent：", messages)
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
        print("已将字符串消息转换为列表格式")

    # 检查并在必要时添加系统提示词
    if not any(
        isinstance(message, dict) and message.get("role") == "system" for message in messages
    ):
        system_prompt = {
            "role": "system",
            "content": "你是乐于助人的助手，能回答关于 Store Sales Price Elasticity Promotions 数据集的问题。",
        }
        messages.append(system_prompt)
        print("已向消息添加系统提示词")

    while True:
        # 路由器调用 span
        print("正在启动路由器调用 span")
        with tracer.start_as_current_span(
            "router_call",
            openinference_span_kind="chain",
        ) as span:
            span.set_input(value=messages)

            response = client.chat.completions.create(
                model=model,
                messages=messages,
                tools=tools,
            )

            messages.append(response.choices[0].message.model_dump())
            tool_calls = response.choices[0].message.tool_calls
            print("收到包含工具调用的回复：", bool(tool_calls))
            span.set_status(StatusCode.OK)

            if tool_calls:
                # 工具调用 span
                print("正在处理工具调用")
                messages = handle_tool_calls(tool_calls, messages)
                span.set_output(value=tool_calls)
            else:
                print("没有工具调用，返回最终回复")
                span.set_output(value=response.choices[0].message.content)

                return response.choices[0].message.content

## 运行 Agent

Agent 已准备就绪！用几个示例问题试一下：

In [32]:
ret = start_main_span(
    [{"role": "user", "content": "Create a dot plot chart showing sales in 2021"}]
)
print(Markdown(ret))

Starting main span with messages: [{'role': 'user', 'content': 'Create a dot plot chart showing sales in 2021'}]
Running agent with messages: [{'role': 'user', 'content': 'Create a dot plot chart showing sales in 2021'}]
Added system prompt to messages
Starting router call span
Received response with tool calls: True
Processing tool calls
Index(['Store_Number', 'SKU_Coded', 'Product_Class_Code', 'Sold_Date',
       'Qty_Sold', 'Total_Sale_Value', 'On_Promo'],
      dtype='object')
sales
Starting router call span
Received response with tool calls: True
Processing tool calls
Starting router call span
Received response with tool calls: False
No tool calls, returning final response
Main span completed with return value: I have created a dot plot chart showing sales data for the year 2021. The chart displays the total quantity sold over the specified dates. Here is the visualization code you can run in your environment:

```python
import pandas as pd
import matplotlib.pyplot as plt
import j

In [ ]:
agent_questions = [
    "What was the most popular product SKU?",
    "What was the total revenue across all stores?",
    "Which store had the highest sales volume?",
    "Create a bar chart showing total sales by store",
    "What percentage of items were sold on promotion?",
    "Plot daily sales volume over time",
    "What was the average transaction value?",
    "Create a box plot of transaction values",
    "Which products were frequently purchased together?",
    "Plot a line graph showing the sales trend over time with a 7-day moving average",
]

for question in tqdm(agent_questions, desc="Processing questions"):
    try:
        ret = start_main_span([{"role": "user", "content": question}])
    except Exception as e:
        print(f"处理问题出错： {question}")
        print(e)
        continue

![Agent 追踪](https://storage.googleapis.com/arize-phoenix-assets/assets/images/agent-traces.png)

# 评估 Agent

Agent 看起来能够工作，但该如何衡量其表现？

In [33]:
OpenAIInstrumentor().uninstrument()  # 取消 OpenAI 客户端插桩，避免将 LLM 裁判评估调用捕获到同一项目。

In [ ]:
import nest_asyncio
from phoenix.client import Client
from phoenix.client.__generated__.v1 import DatasetExample as Example
from phoenix.client.experiments import create_evaluator, evaluate_experiment, run_experiment
from phoenix.client.types.spans import SpanQuery
from phoenix.evals import LLM, ClassificationEvaluator, async_evaluate_dataframe, bind_evaluator
from phoenix.evals.metrics import ToolInvocationEvaluator, ToolSelectionEvaluator

nest_asyncio.apply()

In [ ]:
px_client = Client()
eval_model = LLM(provider="openai", model="gpt-4o-mini")

In [36]:
AVAILABLE_TOOLS = "generate_visualization, lookup_sales_data, analyze_sales_data, run_python_code"

## 使用 LLM 裁判评估函数调用

第一项评估使用另一个 LLM 评估 Agent 路由器的选择。

它遵循标准流程：
1. 从 Phoenix 导出追踪
2. 将导出的追踪整理成具有正确列的 DataFrame
3. 使用 `async_evaluate_dataframe`，结合 `ToolSelectionEvaluator` 和 `ToolInvocationEvaluator` 对每行分类并生成评估标签
4. 将结果上传回 Phoenix

In [ ]:
query = (
    SpanQuery()
    .where(
        "span_kind == 'LLM'",
    )
    .select("input.value", "llm.output_messages")
)

# Phoenix Client 可接收此查询并返回 DataFrame。
tool_calls_df = Client().spans.get_spans_dataframe(
    query=query, project_name=project_name, timeout=None
)
tool_calls_df = tool_calls_df.rename(
    columns={"input.value": "question", "llm.output_messages": "output_messages"}
)
tool_calls_df.dropna(subset=["output_messages"], inplace=True)


def get_tool_call(outputs):
    if outputs[0].get("message").get("tool_calls"):
        return (
            outputs[0]
            .get("message")
            .get("tool_calls")[0]
            .get("tool_call")
            .get("function")
            .get("name")
        )
    else:
        return "No tool used"


tool_calls_df["tool_call"] = tool_calls_df["output_messages"].apply(get_tool_call)
tool_calls_df.head()

In [ ]:
tool_calls_df["available_tools"] = AVAILABLE_TOOLS
tool_calls_df["tool_selection"] = tool_calls_df["tool_call"]

tool_selection_evaluator = ToolSelectionEvaluator(llm=eval_model)
tool_invocation_evaluator = ToolInvocationEvaluator(llm=eval_model)

tool_call_eval = await async_evaluate_dataframe(
    dataframe=tool_calls_df,
    evaluators=[tool_selection_evaluator, tool_invocation_evaluator],
    concurrency=10,
)

tool_call_eval.head()

In [ ]:
from phoenix.client import AsyncClient

px_client = AsyncClient()
await px_client.spans.log_span_annotations_dataframe(
    dataframe=tool_call_eval, annotation_name="Tool Calling Eval", annotator_kind="LLM"
)

现在应该能在 Phoenix 中看到评估标签。

# ![函数调用评估](https://storage.googleapis.com/arize-phoenix-assets/assets/images/function-calling-evals.png)

### 使用真实标签评估函数调用

上述示例可以工作；但如果你有真实标注数据，可以通过运行实验更准确地衡量路由器表现。

Phoenix 中的实验也遵循标准步骤：
1. 创建包含测试用例的数据集，可选提供预期输出
2. 创建对每个测试用例执行的任务，通常是调用 Agent 或其某个步骤
3. 创建对任务的每个输出运行的评估器
4. 在 Phoenix 中查看结果

In [ ]:
import uuid

id = str(uuid.uuid4())

agent_tool_responses = {
    "What was the most popular product SKU?": "lookup_sales_data, analyze_sales_data",
    "What was the total revenue across all stores?": "lookup_sales_data, analyze_sales_data",
    "Which store had the highest sales volume?": "lookup_sales_data, analyze_sales_data",
    "Create a bar chart showing total sales by store": "generate_visualization, lookup_sales_data, run_python_code",
    "What percentage of items were sold on promotion?": "lookup_sales_data, analyze_sales_data",
    "Plot daily sales volume over time": "generate_visualization, lookup_sales_data, run_python_code",
    "What was the average transaction value?": "lookup_sales_data, analyze_sales_data",
    "Create a box plot of transaction values": "generate_visualization, lookup_sales_data, run_python_code",
    "Which products were frequently purchased together?": "lookup_sales_data, analyze_sales_data",
    "Plot a line graph showing the sales trend over time with a 7-day moving average": "generate_visualization, lookup_sales_data, run_python_code",
}

tool_calling_df = pd.DataFrame(agent_tool_responses.items(), columns=["question", "tool_calls"])
dataset = px_client.datasets.create_dataset(
    dataframe=tool_calling_df,
    name=f"tool_calling_ground_truth_{id}",
    input_keys=["question"],
    output_keys=["tool_calls"],
)

任务可以只运行 Agent 的路由器调用：

In [34]:
def run_router_step(example: Example) -> str:
    messages = [
        {
            "role": "system",
            "content": "你是乐于助人的助手，能回答关于 Store Sales Price Elasticity Promotions 数据集的问题。",
        }
    ]
    messages.append({"role": "user", "content": example.input.get("question")})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
    )
    tool_calls = []
    for tool_call in response.choices[0].message.tool_calls:
        tool_calls.append(tool_call.function.name)
    return tool_calls

由于已有预期输出，评估器也可以很简单。没有预期输出时，可以改用 LLM 裁判，或使用基本代码：

In [35]:
def tools_match(expected: str, output: str) -> bool:
    expected_tools = expected.get("tool_calls").split(", ")
    return expected_tools == output

In [ ]:
experiment = run_experiment(
    dataset=dataset,
    task=run_router_step,
    evaluators=[tools_match],
    experiment_name="Tool Calling Eval",
    experiment_description="评估 Agent 的工具调用步骤",
)

## 工具评估

接下来评估 Agent 的工具。每个工具通常采用不同的评估方式，下面提供了一些示例。如需其他思路，可从 [Phoenix 内置评估器](https://arize.com/docs/phoenix/evaluation/how-to-evals/running-pre-tested-evals) 中了解可用的其他指标。

### 评估 SQL 生成工具

In [ ]:
# 此步骤应改用人工标注的真实数据，而非生成示例

db_lookup_questions = [
    "What was the most popular product SKU?",
    "Which store had the highest total sales value?",
    "How many items were sold on promotion?",
    "What was the average quantity sold per transaction?",
    "Which product class code generated the most revenue?",
    "What day of the week had the highest sales volume?",
    "How many unique stores made sales?",
    "What was the highest single transaction value?",
    "Which products were frequently sold together?",
    "What's the trend in sales over time?",
]

expected_results = []

for question in tqdm(db_lookup_questions, desc="正在处理 SQL 查询问题"):
    try:
        with suppress_tracing():
            expected_results.append(lookup_sales_data(question))
    except Exception as e:
        print(f"处理问题出错： {question}")
        print(e)
        db_lookup_questions.remove(question)

# 使用问题创建 DataFrame
questions_df = pd.DataFrame({"question": db_lookup_questions, "expected_result": expected_results})

questions_df

In [ ]:
dataset = px_client.datasets.create_dataset(
    dataframe=questions_df,
    name=f"sales_db_lookup_questions_{id}",
    input_keys=["question"],
    output_keys=["expected_result"],
)

In [40]:
def run_sql_query(example: Example) -> str:
    with suppress_tracing():
        return lookup_sales_data(example.input.get("question"))

In [41]:
def evaluate_sql_result(output: str, expected: str) -> bool:
    # 仅从两个字符串中提取数字
    result_nums = "".join(filter(str.isdigit, output))
    expected_nums = "".join(filter(str.isdigit, expected.get("expected_result")))
    return result_nums == expected_nums

In [ ]:
experiment = run_experiment(
    dataset=dataset,
    task=run_sql_query,
    evaluators=[evaluate_sql_result],
    experiment_name="SQL Query Eval",
    experiment_description="评估 Agent 的 SQL 查询生成步骤",
)

### 评估 Python 代码生成工具

In [ ]:
# 将此处替换为人工标注的真实数据，而非生成示例

code_generation_questions = [
    "Create a bar chart showing total sales by store",
    "Plot daily sales volume over time",
    "Plot a line graph showing the sales trend over time with a 7-day moving average",
    "Create a histogram of quantities sold per transaction",
    "Generate a pie chart showing sales distribution across product classes",
    "Create a stacked bar chart showing promotional vs non-promotional sales by store",
    "Generate a heatmap of sales by day of week and store number",
    "Plot a line chart comparing sales trends between top 5 stores",
]

example_data = []
chart_configs = []
for question in tqdm(code_generation_questions[:], desc="正在处理代码生成问题"):
    try:
        with suppress_tracing():
            example_data.append(lookup_sales_data(question))
            chart_configs.append(json.dumps(extract_chart_config(example_data[-1], question)))
    except Exception as e:
        print(f"处理问题出错： {question}")
        print(e)
        code_generation_questions.remove(question)

code_generation_df = pd.DataFrame(
    {
        "question": code_generation_questions,
        "example_data": example_data,
        "chart_configs": chart_configs,
    }
)

dataset = px_client.datasets.create_dataset(
    dataframe=code_generation_df,
    name=f"code_generation_questions_{id}",
    input_keys=["question", "example_data", "chart_configs"],
)

In [45]:
def run_code_generation(example: Example) -> str:
    with suppress_tracing():
        chart_config = extract_chart_config(
            data=example.input.get("example_data"), visualization_goal=example.input.get("question")
        )
        code = generate_visualization(
            visualization_goal=example.input.get("question"), data=example.input.get("example_data")
        )

    return {"code": code, "chart_config": chart_config}

这里没有可比较的真实标签数据，因此可以使用简单的代码评估器：尝试运行生成的代码并捕获错误。

In [46]:
def code_is_runnable(output: str) -> bool:
    """检查代码是否可运行"""
    output = output.get("code")
    output = output.strip()
    output = output.replace("```python", "").replace("```", "")
    try:
        exec(output)
        return True
    except Exception:
        return False

In [48]:
def evaluate_chart_config(output: str, expected: str) -> bool:
    return output.get("chart_config") == expected.get("chart_config")

In [ ]:
experiment = run_experiment(
    dataset=dataset,
    task=run_code_generation,
    evaluators=[code_is_runnable, evaluate_chart_config],
    experiment_name="Code Generation Eval",
    experiment_description="评估 Agent 的代码生成步骤",
)

# 评估 Agent 路径与收敛性

最后评估 Agent 的执行路径，以了解其执行效率。它是否需要多次调用同一个工具？是否跳过不该跳过的步骤，之后又被迫回退？收敛性或路径评估可以回答这些问题。

收敛性评估的操作略有不同。下面的方法依赖于知道 Agent 处理给定类型查询所需的最少步骤数。我们先运行实验，再追加第二个评估器计算收敛性。

流程如下：
1. 创建同类问题的数据集，每次采用不同表述；Agent 理应为这些问题采用相同路径，但实际往往并非如此。
2. 创建在每个问题上运行 Agent 的任务，同时跟踪步骤数。
3. 不带评估器运行实验。
4. 计算完成任务所需的最少步骤数。
5. 创建评估器，将每次运行的步骤数与最小值比较。
6. 在第 3 步的实验上运行此评估器。
7. 在 Phoenix 中查看结果。

In [ ]:
# 将此处替换为人工标注的真实数据，而非生成示例

convergence_questions = [
    "What was the average quantity sold per transaction?",
    "What is the mean number of items per sale?",
    "Calculate the typical quantity per transaction",
    "Show me the average number of units sold in each transaction",
    "What's the mean transaction size in terms of quantity?",
    "On average, how many items were purchased per transaction?",
    "What is the average basket size per sale?",
    "Calculate the mean number of products per purchase",
    "What's the typical number of units per order?",
    "Find the average quantity of items in each transaction",
    "What is the average number of products bought per purchase?",
    "Tell me the mean quantity of items in a typical transaction",
    "How many items does a customer buy on average per transaction?",
    "What's the usual number of units in each sale?",
    "Calculate the average basket quantity per order",
    "What is the typical amount of products per transaction?",
    "Show the mean number of items customers purchase per visit",
    "What's the average quantity of units per shopping trip?",
    "How many products do customers typically buy in one transaction?",
    "What is the standard basket size in terms of quantity?",
]

convergence_df = pd.DataFrame({"question": convergence_questions})

dataset = px_client.datasets.create_dataset(
    dataframe=convergence_df, name="convergence_questions", input_keys=["question"]
)

In [29]:
def format_message_steps(messages):
    """
    将消息对象列表转换为展示执行步骤的可读格式。

    参数：
        messages (list)：包含角色、内容、工具调用等信息的消息对象列表。

    返回：
        str：展示执行步骤的可读字符串。
    """
    steps = []
    for message in messages:
        role = message.get("role")
        if role == "user":
            steps.append(f"User: {message.get('content')}")
        elif role == "system":
            steps.append("系统：提供上下文")
        elif role == "assistant":
            if message.get("tool_calls"):
                for tool_call in message["tool_calls"]:
                    tool_name = tool_call["function"]["name"]
                    steps.append(f"助手：调用工具 '{tool_name}'")
            else:
                steps.append(f"Assistant: {message.get('content')}")
        elif role == "tool":
            steps.append(f"工具响应： {message.get('content')}")

    return "\n".join(steps)

In [30]:
def run_agent_and_track_path(example: Example) -> str:
    print("使用以下消息启动主 span：", example.input.get("question"))
    messages = [{"role": "user", "content": example.input.get("question")}]
    ret = run_agent_messages(messages)
    return {"path_length": len(ret), "messages": format_message_steps(ret)}


def run_agent_messages(messages):
    print("使用以下消息运行 Agent：", messages)
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
        print("已将字符串消息转换为列表格式")

    # 检查并在必要时添加系统提示词
    if not any(
        isinstance(message, dict) and message.get("role") == "system" for message in messages
    ):
        system_prompt = {
            "role": "system",
            "content": "你是乐于助人的助手，能回答关于 Store Sales Price Elasticity Promotions 数据集的问题。",
        }
        messages.append(system_prompt)
        print("已向消息添加系统提示词")

    while True:
        # 路由器调用 span
        print("Starting router")

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
        )

        messages.append(response.choices[0].message.model_dump())
        tool_calls = response.choices[0].message.tool_calls
        print("收到包含工具调用的回复：", bool(tool_calls))

        if tool_calls:
            # 工具调用 span
            print("正在处理工具调用")
            tool_calls = response.choices[0].message.tool_calls
            messages = handle_tool_calls(tool_calls, messages)
        else:
            print("没有工具调用，返回最终回复")
            return messages

In [ ]:
experiment = run_experiment(
    dataset=dataset,
    task=run_agent_and_track_path,
    experiment_name="Convergence Eval",
    experiment_description="评估 Agent 的收敛性",
)

In [ ]:
experiment.as_dataframe()

In [ ]:
outputs = experiment.as_dataframe()["output"].to_dict().values()
optimal_path_length = min(
    output.get("path_length")
    for output in outputs
    if output and output.get("path_length") is not None
)
print(f"最优路径长度为 {optimal_path_length}")

In [33]:
@create_evaluator(name="Convergence Eval", kind="CODE")
def evaluate_path_length(output: str) -> float:
    if output and output.get("path_length"):
        return optimal_path_length / float(output.get("path_length"))
    else:
        return 0

In [ ]:
experiment = evaluate_experiment(experiment=experiment, evaluators=[evaluate_path_length])

# 进阶：将所有评估合并到一个实验

作为可选的最后一步，你可以把上述所有评估器和实验合并成一个实验。这需要更复杂的数据整理，但能生成统一的 Agent 性能报告。

### 构建能记录评估所需全部信息的 Agent 版本

In [72]:
def process_messages(messages):
    tool_calls = []
    tool_responses = []
    final_output = None

    for i, message in enumerate(messages):
        # 提取工具调用
        if "tool_calls" in message and message["tool_calls"]:
            for tool_call in message["tool_calls"]:
                tool_name = tool_call["function"]["name"]
                tool_input = tool_call["function"]["arguments"]
                tool_calls.append(tool_name)

                # 准备包含工具名与输入的工具响应结构
                tool_responses.append(
                    {"tool_name": tool_name, "tool_input": tool_input, "tool_response": None}
                )

        # 提取工具响应
        if message["role"] == "tool" and "tool_call_id" in message:
            for tool_response in tool_responses:
                if message["tool_call_id"] in message.values():
                    tool_response["tool_response"] = message["content"]

        # 提取最终输出
        if (
            message["role"] == "assistant"
            and not message.get("tool_calls")
            and not message.get("function_call")
        ):
            final_output = message["content"]

    result = {
        "tool_calls": tool_calls,
        "tool_responses": tool_responses,
        "final_output": final_output,
        "unchanged_messages": messages,
        "path_length": len(messages),
    }

    return result

In [73]:
def run_agent_and_track_path_combined(example: Example) -> str:
    print("使用以下消息启动主 span：", example.input.get("question"))
    messages = [{"role": "user", "content": example.input.get("question")}]
    ret = run_agent_messages_combined(messages)
    return process_messages(ret)


def run_agent_messages_combined(messages):
    print("使用以下消息运行 Agent：", messages)
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
        print("已将字符串消息转换为列表格式")

    # 检查并在必要时添加系统提示词
    if not any(
        isinstance(message, dict) and message.get("role") == "system" for message in messages
    ):
        system_prompt = {
            "role": "system",
            "content": "你是乐于助人的助手，能回答关于 Store Sales Price Elasticity Promotions 数据集的问题。",
        }
        messages.append(system_prompt)
        print("已向消息添加系统提示词")

    while True:
        # 路由器调用 span
        print("Starting router")

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
        )

        messages.append(response.choices[0].message.model_dump())
        tool_calls = response.choices[0].message.tool_calls
        print("收到包含工具调用的回复：", bool(tool_calls))

        if tool_calls:
            # 工具调用 span
            print("正在处理工具调用")
            tool_calls = response.choices[0].message.tool_calls
            messages = handle_tool_calls(tool_calls, messages)
        else:
            print("没有工具调用，返回最终回复")
            return messages

In [ ]:
generate_sql_query("What was the most popular product SKU?", store_sales_df.columns, "sales")

In [ ]:
overall_experiment_questions = [
    {
        "question": "What was the most popular product SKU?",
        "sql_result": "   SKU_Coded  Total_Qty_Sold 0    6200700         52262.0",
    },
    {
        "question": "What was the total revenue across all stores?",
        "sql_result": "   Total_Revenue 0   1.327264e+07",
    },
    {
        "question": "Which store had the highest sales volume?",
        "sql_result": "   Store_Number  Total_Sales_Volume 0          2970             59322.0",
    },
    {
        "question": "Create a bar chart showing total sales by store",
        "sql_result": "    Store_Number    Total_Sales 0            880  420302.088397 1           1650  580443.007953 2           4180  272208.118542 3            550  229727.498752 4           1100  497509.528013 5           3300  619660.167018 6           3190  335035.018792 7           2970  836341.327191 8           3740  359729.808228 9           2530  324046.518720 10          4400   95745.620250 11          1210  508393.767785 12           330  370503.687331 13          2750  453664.808068 14          1980  242290.828499 15          1760  350747.617798 16          3410  410567.848126 17           990  378433.018639 18          4730  239711.708869 19          4070  322307.968330 20          3080  495458.238811 21          2090  309996.247965 22          1320  592832.067579 23          2640  308990.318559 24          1540  427777.427815 25          4840  389056.668316 26          2860  132320.519487 27          2420  406715.767402 28           770  292968.918642 29          3520  145701.079372 30           660  343594.978075 31          3630  405034.547846 32          2310  412579.388504 33          2200  361173.288199 34          1870  401070.997685",
    },
    {
        "question": "What percentage of items were sold on promotion?",
        "sql_result": "   Promotion_Percentage 0              0.625596",
    },
    {
        "question": "What was the average transaction value?",
        "sql_result": "   Average_Transaction_Value 0                  19.018132",
    },
    {
        "question": "Create a line chart showing sales in 2021",
        "sql_result": "  sale_month  total_quantity_sold  total_sales_value 0 2021-11-01              43056.0      499984.428193 1 2021-12-01              75724.0      910982.118423",
    },
]

overall_experiment_questions[0]["sql_generated"] = generate_sql_query(
    overall_experiment_questions[0]["question"], store_sales_df.columns, "sales"
)
overall_experiment_questions[1]["sql_generated"] = generate_sql_query(
    overall_experiment_questions[1]["question"], store_sales_df.columns, "sales"
)
overall_experiment_questions[2]["sql_generated"] = generate_sql_query(
    overall_experiment_questions[2]["question"], store_sales_df.columns, "sales"
)
overall_experiment_questions[3]["sql_generated"] = generate_sql_query(
    overall_experiment_questions[3]["question"], store_sales_df.columns, "sales"
)
overall_experiment_questions[4]["sql_generated"] = generate_sql_query(
    overall_experiment_questions[4]["question"], store_sales_df.columns, "sales"
)
overall_experiment_questions[5]["sql_generated"] = generate_sql_query(
    overall_experiment_questions[5]["question"], store_sales_df.columns, "sales"
)
overall_experiment_questions[6]["sql_generated"] = generate_sql_query(
    overall_experiment_questions[6]["question"], store_sales_df.columns, "sales"
)

print(overall_experiment_questions[6])

# overall_experiment_df = pd.DataFrame(overall_experiment_questions)

# dataset = px_client.datasets.create_dataset(dataframe=overall_experiment_df, name="overall_experiment_questions_all", input_keys=["question"], output_keys=["sql_result"])

In [ ]:
print(overall_experiment_questions[6])

In [ ]:
[
    {
        "question": "What was the most popular product SKU?",
        "sql_result": "   SKU_Coded  Total_Qty_Sold 0    6200700         52262.0",
        "sql_generated": "```sql\nSELECT SKU_Coded, SUM(Qty_Sold) AS Total_Qty_Sold\nFROM sales\nGROUP BY SKU_Coded\nORDER BY Total_Qty_Sold DESC\nLIMIT 1;\n```",
    },
    {
        "question": "What was the total revenue across all stores?",
        "sql_result": "   Total_Revenue 0   1.327264e+07",
        "sql_generated": "```sql\nSELECT SUM(Total_Sale_Value) AS Total_Revenue\nFROM sales;\n```",
    },
    {
        "question": "Which store had the highest sales volume?",
        "sql_result": "   Store_Number  Total_Sales_Volume 0          2970             59322.0",
        "sql_generated": "```sql\nSELECT Store_Number, SUM(Total_Sale_Value) AS Total_Sales_Volume\nFROM sales\nGROUP BY Store_Number\nORDER BY Total_Sales_Volume DESC\nLIMIT 1;\n```",
    },
    {
        "question": "Create a bar chart showing total sales by store",
        "sql_result": "    Store_Number    Total_Sales 0            880  420302.088397 1           1650  580443.007953 2           4180  272208.118542 3            550  229727.498752 4           1100  497509.528013 5           3300  619660.167018 6           3190  335035.018792 7           2970  836341.327191 8           3740  359729.808228 9           2530  324046.518720 10          4400   95745.620250 11          1210  508393.767785 12           330  370503.687331 13          2750  453664.808068 14          1980  242290.828499 15          1760  350747.617798 16          3410  410567.848126 17           990  378433.018639 18          4730  239711.708869 19          4070  322307.968330 20          3080  495458.238811 21          2090  309996.247965 22          1320  592832.067579 23          2640  308990.318559 24          1540  427777.427815 25          4840  389056.668316 26          2860  132320.519487 27          2420  406715.767402 28           770  292968.918642 29          3520  145701.079372 30           660  343594.978075 31          3630  405034.547846 32          2310  412579.388504 33          2200  361173.288199 34          1870  401070.997685",
        "sql_generated": "```sql\nSELECT Store_Number, SUM(Total_Sale_Value) AS Total_Sales\nFROM sales\nGROUP BY Store_Number;\n```",
    },
    {
        "question": "What percentage of items were sold on promotion?",
        "sql_result": "   Promotion_Percentage 0              0.625596",
        "sql_generated": "```sql\nSELECT \n    (SUM(CASE WHEN On_Promo = 'Yes' THEN 1 ELSE 0 END) * 100.0) / COUNT(*) AS Promotion_Percentage\nFROM \n    sales;\n```",
    },
    {
        "question": "What was the average transaction value?",
        "sql_result": "   Average_Transaction_Value 0                  19.018132",
        "sql_generated": "```sql\nSELECT AVG(Total_Sale_Value) AS Average_Transaction_Value\nFROM sales;\n```",
    },
    {
        "question": "Create a line chart showing sales in 2021",
        "sql_result": "  sale_month  total_quantity_sold  total_sales_value 0 2021-11-01              43056.0      499984.428193 1 2021-12-01              75724.0      910982.118423",
        "sql_generated": "```sql\nSELECT MONTH(Sold_Date) AS Month, SUM(Total_Sale_Value) AS Total_Sales\nFROM sales\nWHERE YEAR(Sold_Date) = 2021\nGROUP BY MONTH(Sold_Date)\nORDER BY MONTH(Sold_Date);\n```",
    },
]

In [65]:
CLARITY_LLM_JUDGE_PROMPT = """
本任务将提供查询和答案，你的目标是评估答案回应查询时的清晰度。
清晰回答应精确、连贯、直接处理查询，不引入不必要的复杂性或歧义。
不清晰回答则含糊、混乱或难以理解，即使事实可能正确。

回复应是单个词："clear" 或 "unclear"，不得包含其他文字或字符。
"clear" 表示答案结构良好、易于理解并恰当回应查询。
"unclear" 表示答案模糊、组织不佳或表达无效。
请仔细考虑查询和答案后再作判断。

分析查询和答案后，必须详细解释推理，说明选择 "clear" 或 "unclear" 的理由。
不要在解释开头给出最终标签。推理应包含具体要点，说明答案是否符合清晰度标准。

[BEGIN DATA]
Query: {query}
Answer: {response}
[END DATA]
请仔细分析数据，先解释，再给出判断。

EXPLANATION: 逐步说明推理，根据查询评估答案清晰度。
LABEL: "clear" 或 "unclear"
"""

ENTITY_CORRECTNESS_LLM_JUDGE_PROMPT = """
本任务将提供查询和答案。你的目标是判断答案中提到的所有实体是否正确识别，
并准确匹配查询中的实体。实体指具体人物、地点、组织、日期或其他专有名词。
重点检查答案中的实体名称是否正确，以及与查询上下文的关联是否恰当。

回复应是单个词："correct" 或 "incorrect"，不得包含其他文字或字符。
"correct" 表示答案中所有实体都与查询匹配且识别正确。
"incorrect" 表示答案中的实体相对于查询存在错误或不匹配。

分析查询和答案后，必须详细解释推理，说明选择 "correct" 或 "incorrect" 的理由。
不要在解释开头给出最终标签。推理应包含具体要点，说明答案实体是否匹配查询实体。

[BEGIN DATA]
Query: {query}
Answer: {response}
[END DATA]
请仔细分析数据，先解释，再给出判断。

EXPLANATION: 逐步说明推理，评估答案实体是否正确且与查询一致。
LABEL: "correct" 或 "incorrect"
"""

In [40]:
# 供实验评估器使用的 JSON 格式可用工具
available_tools_str = json.dumps(tools)

In [ ]:
tool_selection_eval = ToolSelectionEvaluator(llm=eval_model)
tool_invocation_eval = ToolInvocationEvaluator(llm=eval_model)


async def function_calling_eval(input: str, output: str) -> float:
    """评估完整回复中的工具选择与调用。

    这里封装内置评估器，因为必须先从输出中提取 tool_calls，
    并整理为人类可读字符串，才能传入评估器提示词。
    """
    function_calls = output.get("tool_calls")
    if not function_calls:
        return 0

    tool_selection = ", ".join(function_calls)
    selection_result = tool_selection_eval.evaluate(
        eval_input={
            "input": input.get("question"),
            "tool_selection": tool_selection,
            "available_tools": available_tools_str,
        }
    )
    invocation_result = tool_invocation_eval.evaluate(
        eval_input={
            "input": input.get("question"),
            "tool_selection": tool_selection,
            "available_tools": available_tools_str,
        }
    )
    return float(selection_result[0].label == "correct" and invocation_result[0].label == "correct")


def code_is_runnable(output: str) -> bool:
    """检查代码是否可运行"""
    generated_code = output.get("tool_responses")
    if not generated_code:
        return True

    # 查找第一条 lookup_sales_data 响应
    generated_code = next(
        (r for r in generated_code if r.get("tool_name") == "generate_visualization"), None
    )
    if not generated_code:
        return True

    # 获取第一条响应
    generated_code = generated_code.get("tool_response", "")
    generated_code = generated_code.strip()
    generated_code = generated_code.replace("```python", "").replace("```", "")
    try:
        exec(generated_code)
        return True
    except Exception:
        return False


def evaluate_sql_result(output, expected) -> bool:
    sql_result = output.get("tool_responses")
    if not sql_result:
        return True

    # 查找第一条 lookup_sales_data 响应
    sql_result = next((r for r in sql_result if r.get("tool_name") == "lookup_sales_data"), None)
    if not sql_result:
        return True

    # 获取第一条响应
    sql_result = sql_result.get("tool_response", "")

    # 仅从两个字符串中提取数字
    result_nums = "".join(filter(str.isdigit, sql_result))
    expected_nums = "".join(filter(str.isdigit, expected.get("sql_result")))
    return result_nums == expected_nums


clarity_evaluator = bind_evaluator(
    evaluator=ClassificationEvaluator(
        name="clarity",
        prompt_template=CLARITY_LLM_JUDGE_PROMPT,
        llm=eval_model,
        choices={"clear": 1.0, "unclear": 0.0},
    ),
    input_mapping={
        "query": "input.question",
        "response": "output.final_output",
    },
)

entity_evaluator = bind_evaluator(
    evaluator=ClassificationEvaluator(
        name="entity_correctness",
        prompt_template=ENTITY_CORRECTNESS_LLM_JUDGE_PROMPT,
        llm=eval_model,
        choices={"correct": 1.0, "incorrect": 0.0},
    ),
    input_mapping={
        "query": "input.question",
        "response": "output.final_output",
    },
)

In [ ]:
def run_overall_experiment(example: Example) -> str:
    with suppress_tracing():
        return run_agent_and_track_path_combined(example)


experiment = run_experiment(
    dataset=dataset,
    task=run_overall_experiment,
    evaluators=[
        function_calling_eval,
        evaluate_sql_result,
        clarity_evaluator,
        entity_evaluator,
        code_is_runnable,
    ],
    experiment_name="Overall Experiment",
    experiment_description="评估整体实验",
)

## 恭喜！🎉

你已经评估了 Agent 的各个方面。完成本教程后，你已掌握评估 Agent 路由器、工具和执行路径的方法！

# ![合并的 Agent 实验](https://storage.googleapis.com/arize-phoenix-assets/assets/images/combined-agent-experiments.png)
